# Retrieval-Augmented Generation (RAG) System for Legal Question Answering

## Dataset Description

This project uses a legal dataset from Kaggle : https://www.kaggle.com/datasets/bharathpriyankumar/the-indian-legal-database-nlprag

The dataset contains information about Indian law and constitutional rules.

Each entry includes :
- The original legal text
- A simplified explanation to make it easier to understand

This helps the model :
- find relevant legal information
- generate clear and simple answers

The dataset is well structured and suitable for building a RAG system.
This structure allows the system to retrieve relevant legal information and generate answers that are both accurate and easy to understand.

In [1]:
#%pip install sentence-transformers chromadb langchain-text-splitters transformers

#### Step 1 - Load Data

In [2]:
import pandas as pd

df = pd.read_csv("Indian_Law_SupremeCourt_Database.csv")
df.head()

,provision_id,source_type,part_chapter,article_section,title,verbatim_text_excerpt,simplified_explanation,keywords,landmark_case_1,lc1_year,...,nodal_ministry,legal_classification,punishment_quantum,exceptions_and_limitations,cross_references,bench_strength,current_precedent_status,primary_source_url,current_law_mapping,row_status
0,IND-001,Constitution,Preamble,Preamble,Preamble of India,"WE, THE PEOPLE OF INDIA, having solemnly resol...",The Preamble is the introduction to the Consti...,"Preamble,Sovereign,Socialist,Secular,Democrati...",Kesavananda Bharati v. State of Kerala,1973.0,...,Ministry of Law and Justice / Respective Ministry,Not applicable,Not applicable,Subject to applicable constitutional/statutory...,Related constitutional/statutory provisions as...,Legislative Enactment,In force,https://www.indiacode.nic.in/ / Supreme Court ...,Constitutional / Statutory Mandate,Active
1,IND-002,Constitution,Part I - The Union and its Territory,Article 1,Name and Territory of the Union,"India, that is Bharat, shall be a Union of States",India is officially called India or Bharat. It...,"Union of States,Bharat,India,Territory,Indestr...",In re Berubari Union,1960.0,...,Ministry of Law and Justice / Respective Ministry,Not applicable,Not applicable,Subject to applicable constitutional/statutory...,Related constitutional/statutory provisions as...,Legislative Enactment,In force,https://www.indiacode.nic.in/ / Supreme Court ...,Constitutional / Statutory Mandate,Active
2,IND-003,Constitution,Part I - The Union and its Territory,Article 3,Formation of New States,Parliament may by law form a new State by sepa...,Parliament alone has the power to create new s...,"New States,Bifurcation,Reorganisation,Parliame...",Babulal Parate v. State of Bombay,1960.0,...,Ministry of Law and Justice / Respective Ministry,Not applicable,Not applicable,Subject to applicable constitutional/statutory...,Related constitutional/statutory provisions as...,Legislative Enactment,In force,https://www.indiacode.nic.in/ / Supreme Court ...,Constitutional / Statutory Mandate,Active
3,IND-004,Constitution,Part II - Citizenship,Article 5,Citizenship at commencement of Constitution,"At the commencement of this Constitution, ever...",At the time the Constitution came into force (...,"Citizenship,Domicile,Birth,Commencement",Izhar Ahmad Khan v. Union of India,1962.0,...,Ministry of Law and Justice / Respective Ministry,Not applicable,Not applicable,Subject to applicable constitutional/statutory...,Related constitutional/statutory provisions as...,Legislative Enactment,In force,https://www.indiacode.nic.in/ / Supreme Court ...,Constitutional / Statutory Mandate,Active
4,IND-005,Constitution,Part III - Fundamental Rights,Article 12,Definition of State,"In this Part, unless the context otherwise req...",'State' for Fundamental Rights purposes includ...,"State,Fundamental Rights,Government,Authority,...",Ajay Hasia v. Khalid Mujib,1981.0,...,Ministry of Law and Justice / Respective Ministry,Not applicable,Not applicable,Subject to applicable constitutional/statutory...,Related constitutional/statutory provisions as...,Legislative Enactment,In force,https://www.indiacode.nic.in/ / Supreme Court ...,Constitutional / Statutory Mandate,Active


#### Step 2 - Creating documents/extracting text

In [3]:
documents = []
for i, row in df.iterrows():
    text = f"""
Law Text:
{row['verbatim_text_excerpt']}

Explanation:
{row['simplified_explanation']}
"""
    documents.append(text)

print("Total documents:", len(documents))

Total documents: 373


#### Step 3 - Chunking

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.create_documents(documents)
print("Total chunks:", len(chunks))

c:\Users\selin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 690


#### Step 4 - Embedding a sentence

In [5]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def emb_text(text):
    return embedding_model.encode(text).tolist()

Loading weights: 100%|██████████| 103/103 [00:00<?, ?it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### Step 5 - Creating a ChromaDB data collection

In [6]:
import chromadb

client = chromadb.Client()
collection = client.create_collection(name="legal_rag")

In [7]:
for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content],
        ids=[str(i)],
        embeddings=[emb_text(chunk.page_content)]
    )

print("Data added")

Data added


#### Step 6 - Retrieving data for a query

In [8]:
question = "Explain Fundamental Rights in simple terms"

query_embedding = emb_text(question)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

retrieved_docs = results["documents"][0]

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---\n")
    print(doc[:300])

context = "\n\n".join([
    doc for doc in retrieved_docs
    if "article 13" not in doc.lower()
    and "udhr" not in doc.lower()
])


--- Document 1 ---

Explanation:
'State' for Fundamental Rights purposes includes not just Central and State governments but also local bodies, statutory corporations, government companies, and any authority exercising governmental functions. This broad definition ensures Fundamental Rights can be enforced against all 

--- Document 2 ---

Law Text:
Fundamental Rights must be construed broadly and liberally. Where a law restricts a Fundamental Right, the restriction must be construed strictly. The right-conferring provision must be given the widest possible interpretation; the restriction-enabling provision must be given the narrowest

--- Document 3 ---

Law Text:
Fundamental Rights are primarily available against the State (defined under Article 12). However, courts have extended their protection against private parties in certain circumstances — especially when the private party is discharging a public function, has a monopoly, or is substantially

--- Document 4 ---

Explanation:


#### Step 7 - Creating a prompt

In [9]:
PROMPT = """
Context:
{context}

Question:
{question}

Answer:
"""

prompt = PROMPT.format(context=context, question=question)

This prompt has no instructions so the model answers freely (less controlled, can hallucinate)

#### Step 8 - LLM

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6131.01it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [11]:
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nANSWER:\n")
print(answer)


ANSWER:

Fundamental Rights are primarily available against the State (defined under Article 12). However, courts have extended their protection against private parties in certain circumstances — especially when the private party is discharging a public function, has a monopoly, or is substantially funded/controlled by the State Law


#### Step 9 - Evaluation

In [12]:
test_questions = [
    "Who do fundamental rights apply to?",
    "When can private parties be affected by rights?",
    "What is equal protection of law?",
    "What is reasonable classification?",
    "What does equality before law mean?"
]

for q in test_questions:
    # same pipeline as before
    query_embedding = emb_text(q)
    results = collection.query(query_embeddings=[query_embedding], n_results=5)
    retrieved_docs = results["documents"][0]

    context = "\n\n".join(retrieved_docs[:2])
    prompt = PROMPT.format(context=context, question=q)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=120)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nQUESTION:", q)
    print("ANSWER:", answer)
    print("-"*50)


QUESTION: Who do fundamental rights apply to?
ANSWER: the State
--------------------------------------------------

QUESTION: When can private parties be affected by rights?
ANSWER: when the private party is discharging a public function, has a monopoly, or is substantially funded/controlled by the State
--------------------------------------------------

QUESTION: What is equal protection of law?
ANSWER: equals must be treated equally
--------------------------------------------------

QUESTION: What is reasonable classification?
ANSWER: economic criterion alone can be the basis for classification
--------------------------------------------------

QUESTION: What does equality before law mean?
ANSWER: no one is above the law
--------------------------------------------------


#### Step 10 - Testing a new model

In [13]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 4159.34it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [14]:
test_questions = [
    "Who do fundamental rights apply to?",
    "When can private parties be affected by rights?",
    "What is equal protection of law?",
    "What is reasonable classification?",
    "What does equality before law mean?"
]

for q in test_questions:
    query_embedding = emb_text(q)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]

    context = "\n\n".join(retrieved_docs[:2])

    prompt = PROMPT.format(context=context, question=q)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nQUESTION:", q)
    print("ANSWER:", answer)
    print("-"*50)


QUESTION: Who do fundamental rights apply to?
ANSWER: the State
--------------------------------------------------

QUESTION: When can private parties be affected by rights?
ANSWER: in certain circumstances
--------------------------------------------------

QUESTION: What is equal protection of law?
ANSWER: equals must be treated equally
--------------------------------------------------

QUESTION: What is reasonable classification?
ANSWER: economic criterion alone can be the basis for classification
--------------------------------------------------

QUESTION: What does equality before law mean?
ANSWER: no one is above the law
--------------------------------------------------


We compared flan-t5-base and flan-t5-large.
They give very similar answers overall, showing that increasing the model size does not significantly change the results, although flan-t5-large sometimes gives shorter and less precise answers (ex : “in certain circumstances”), while flan-t5-base provides slightly more detailed and informative responses, suggesting that the larger model does not necessarily improve performance for this task.


#### Step 11 - Testing a new prompt

In [15]:
PROMPT2 = """
Answer the question in a simple way using ONLY the context below.

Context:
{context}

Question:
{question}

Answer shortly :
"""

prompt = PROMPT2.format(context=context, question=q)

PROMPT2 forces simple and detailed answers using the context (more structured and explanatory)

In [16]:
test_questions = [
    "Who do fundamental rights apply to?",
    "When can private parties be affected by rights?",
    "What is equal protection of law?",
    "What is reasonable classification?",
    "What does equality before law mean?"
]

for q in test_questions:
    query_embedding = emb_text(q)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]

    context = "\n\n".join(retrieved_docs[:2])

    prompt = PROMPT2.format(context=context, question=q)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nQUESTION:", q)
    print("ANSWER:", answer)
    print("-"*50)


QUESTION: Who do fundamental rights apply to?
ANSWER: the State
--------------------------------------------------

QUESTION: When can private parties be affected by rights?
ANSWER: when the private party is discharging a public function
--------------------------------------------------

QUESTION: What is equal protection of law?
ANSWER: equals must be treated equally
--------------------------------------------------

QUESTION: What is reasonable classification?
ANSWER: economic criterion alone can be the basis for classification
--------------------------------------------------

QUESTION: What does equality before law mean?
ANSWER: no one is above the law
--------------------------------------------------


In [17]:
PROMPT3 = """
You are a strict assistant.
Answer ONLY if the answer is in the context.
If not, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PROMPT3.format(context=context, question=q)

PROMPT3 enforces strict use of context only (safer, may refuse and say “I don’t know” if missing info).

In [18]:
test_questions = [
    "Who do fundamental rights apply to?",
    "When can private parties be affected by rights?",
    "What is equal protection of law?",
    "What is reasonable classification?",
    "What does equality before law mean?"
]

for q in test_questions:
    query_embedding = emb_text(q)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]

    context = "\n\n".join(retrieved_docs[:2])

    prompt = PROMPT3.format(context=context, question=q)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nQUESTION:", q)
    print("ANSWER:", answer)
    print("-"*50)


QUESTION: Who do fundamental rights apply to?
ANSWER: State
--------------------------------------------------

QUESTION: When can private parties be affected by rights?
ANSWER: in certain circumstances
--------------------------------------------------

QUESTION: What is equal protection of law?
ANSWER: equals must be treated equally
--------------------------------------------------

QUESTION: What is reasonable classification?
ANSWER: I don't know
--------------------------------------------------

QUESTION: What does equality before law mean?
ANSWER: no one is above the law
--------------------------------------------------


#### Analysis of the different prompts
PROMPT and PROMPT2 give almost the same answers, which means the extra instructions in PROMPT2 do not really change how the model behaves, while PROMPT3 gives shorter and more cautious answers, sometimes less precise or saying “I don’t know,” because it strictly follows the context but gives less complete responses.

#### Step 12 - User interface

In [ ]:
#%pip install gradio


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
def rag_answer(q):
    query_embedding = emb_text(q)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    retrieved_docs = results["documents"][0]

    context = "\n\n".join(retrieved_docs[:2])

    prompt = PROMPT.format(context=context, question=q)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = model.generate(**inputs, max_new_tokens=120)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

##### Creating the interface

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=rag_answer,
    inputs=gr.Textbox(lines=2, placeholder="Ask a legal question..."),
    outputs="text",
    title="Legal Assistant",
    description="Ask questions about Fundamental Rights"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
